# Colab — srd-facegeom

Un seul notebook pour toutes les étapes. Exécuter **1** à chaque session, puis la cellule de l'étape voulue.

GPU : Exécution → Modifier le type d'exécution → GPU T4 (nécessaire pour l'extraction et les entraînements).


## 1. Mise en place (à chaque session)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/srd-facegeom")   # compte d'Arsène, partagé avec le groupe
CODE_DIR = Path("/content/srd-facegeom")
for sub in ["data", "features", "runs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Arsenicophe/srd-facegeom.git", str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR / "src"))   # évite le redémarrage demandé par pip -e
os.environ["PYTHONPATH"] = str(CODE_DIR / "src")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Seulement ce que Colab n'a pas : réinstaller numpy ou torch casserait Colab
%pip install -q pytorch-metric-learning

import torch
from importlib.metadata import version
for pkg in ["torch", "torchvision", "numpy", "scikit-learn", "pytorch-metric-learning"]:
    print(f"{pkg:24}", version(pkg))
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")


## 2. Images sur le disque local (étapes qui lisent les images seulement, ~2 min)

In [ ]:
from facegeom.data.images import ensure_local_images

IMG_DIR = ensure_local_images(DRIVE_ROOT / "data/celeba/img_align_celeba.zip", "/content/data")
print(IMG_DIR, len(os.listdir(IMG_DIR)), "images")


## Étape 0e — Télécharger CelebA sur Drive (une seule fois, CPU suffit)

In [ ]:
!python scripts/01_download_celeba.py


## Étape 1 — Extraction des vecteurs (GPU ; relancer reprend au dernier morceau)

In [ ]:
LIMIT = None   # 1000 pour un essai dans features_essai1000, None pour l'extraction complète

!python scripts/05_extract_features.py --source {IMG_DIR} {f"--limit {LIMIT}" if LIMIT else ""}


## Étape 2 — Mesures préalables (vecteurs sur Drive)

In [ ]:
FEATURES = DRIVE_ROOT / "features"

!python scripts/06_baselines.py --features-root {FEATURES}


## Étapes 3 et 4 — Entraînement de la tête (GPU)

In [ ]:
LOSS = "arcface"   # nsoftmax, arcface, contrastive, triplet

# Réglage : 5 tirages, graine 100
!python scripts/09_train.py --loss {LOSS} --tune --features-root {FEATURES}

# Graines finales, à partir du best.json du réglage :
# !python scripts/09_train.py --loss {LOSS} --params-from runs/<date>_3_tune_{LOSS}/best.json --features-root {FEATURES}
